# 02 - Preparation des donnees et variables metier

Objectif : transformer les variables brutes en signaux plus adaptes au modele, tout en restant simple a expliquer.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.append(str(BASE_DIR))
from feature_engineering import prepare_customer_features

df = pd.read_csv(BASE_DIR / 'data' / 'customer_churn.csv')
prepared = prepare_customer_features(df)
prepared.head()

## Variables ajoutees

Les nouvelles variables suivent une logique CRM : plainte, paiement, satisfaction, activite, support et revenu.

In [ ]:
new_cols = [c for c in prepared.columns if c not in df.columns]
new_cols

## Exemple d interpretation

- `low_satisfaction` vaut 1 si le client a un NPS negatif, un CSAT faible ou une reponse insatisfaite.
- `inactive_customer` vaut 1 si le client utilise peu le service ou ne s est pas connecte recemment.
- `tickets_per_tenure` evite de comparer directement un nouveau client et un ancien client.

In [ ]:
prepared.groupby('churn')[new_cols].mean().round(3)

## Separation X / y

On retire `customer_id`, car c est un identifiant, et `churn`, car c est la cible.

In [ ]:
X = prepared.drop(columns=['customer_id', 'churn'])
y = prepared['churn']
print(X.shape, y.shape)
print('Colonnes numeriques :', len(X.select_dtypes(exclude='object').columns))
print('Colonnes categorielles :', len(X.select_dtypes(include='object').columns))

## Point important

Ces transformations sont appliquees avant le split uniquement pour creer des colonnes. Les transformations apprenantes comme standardisation et encodage sont ensuite placees dans les pipelines de modele pour eviter le data leakage.